In [30]:
from __future__ import annotations
from dataclasses import dataclass, field
import datetime
import enum
import calendar
import heapq
import random
from typing import Any, Callable, Iterable, Mapping, Protocol, TypeVar

### Validation helpers ###

def _require_non_empty_text(value: str, field_name: str) -> None:
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{field_name} must be a non-empty string.")

def _require_finite_float(value: float, field_name: str) -> None:
    if not isinstance(value, (int, float)) or isinstance(value, bool):
        raise ValueError(f"{field_name} must be a float.")
    if not (float('-inf') < value < float('inf')):
        raise ValueError(f"{field_name} must be a finite float.")

def _require_non_negative_float(value: float, field_name: str) -> None:
    _require_finite_float(value, field_name)
    if value < 0:
        raise ValueError(f"{field_name} must be a non-negative float.")

def _require_probability(value: float, field_name: str) -> None:
    _require_finite_float(value, field_name)
    if not (0 <= value <= 1):
        raise ValueError(f"{field_name} must be a probability (between 0 and 1).")

def _require_day_of_month(value: int, field_name: str) -> None:
    if not isinstance(value, int) or isinstance(value, bool):
        raise ValueError(f"{field_name} must be an integer.")
    if not (1 <= value <= 31):
        raise ValueError(f"{field_name} must be a valid day of the month (1-31).")

def _require_positive_int(value: int, field_name: str) -> None:
    if not isinstance(value, int) or isinstance(value, bool):
        raise ValueError(f"{field_name} must be an integer.")
    if value <= 0:
        raise ValueError(f"{field_name} must be a positive integer.")

### Date and Schedule helpers ###

def monthly_due_date_on_or_after(start: datetime.date, day_of_month: int) -> datetime.date:
    """Return the first date on or after `start` whose day of month is `day_of_month`, clamped to the last day for shorter months."""

    _require_day_of_month(day_of_month, "day_of_month")

    last_day = calendar.monthrange(start.year, start.month)[1]
    candidate = datetime.date(start.year, start.month, min(day_of_month, last_day))

    if candidate < start:
        return next_monthly_due_date(candidate, day_of_month)

    return candidate


def monthly_due_date_after(start: datetime.date, day_of_month: int) -> datetime.date:
    """Return the first monthly due date strictly after `start`."""

    candidate = monthly_due_date_on_or_after(start, day_of_month)

    if candidate <= start:
        return next_monthly_due_date(candidate, day_of_month)
    
    return candidate

def next_monthly_due_date(current_due_date: datetime.date, day_of_month: int) -> datetime.date:
    """
    Return the corresponding due date in the next calendar month, clamped to the last day for shorter months.
    
    Example:
        2026-01-31 -> 2026-02-28 when day_of_month = 31
        2026-02-28 -> 2026-03-31 when day_of_month = 31
    """

    _require_day_of_month(day_of_month, "day_of_month")

    year = current_due_date.year + (current_due_date.month // 12)
    month = current_due_date.month % 12 + 1
    last_day = calendar.monthrange(year, month)[1]
    return datetime.date(year, month, min(day_of_month, last_day))


@dataclass(frozen=True, slots=True)
class CalendarMonthlySchedule:
    """Calendar-month recurrence policy for salaries, rent, mortgages, etc."""

    day_of_month: int

    def __post_init__(self) -> None:
        _require_day_of_month(self.day_of_month, "CalendarMonthlySchedule.day_of_month")
    
    def first_on_or_after(self, start: datetime.date) -> datetime.date:
        return monthly_due_date_on_or_after(start, self.day_of_month)
    
    def first_after(self, start: datetime.date) -> datetime.date:
        return monthly_due_date_after(start, self.day_of_month)
    
    def next_after(self, current_due_date: datetime.date) -> datetime.date:
        return next_monthly_due_date(current_due_date, self.day_of_month)
    
@dataclass(frozen=True, slots=True)
class FixedDayIntervalSchedule:
    """Fixed-day recurrence policy for things that are intentionnaly not tied to calendar months, such as retrying a job search every 30 days."""
    
    days: int
    
    def __post_init__(self) -> None:
        _require_positive_int(self.days, "FixedDayIntervalSchedule.days")
    
    def next_after(self, current_date: datetime.date) -> datetime.date:
        return current_date + datetime.timedelta(days=self.days)

### Domain state ###

@dataclass(frozen=True, slots=True)
class Job:
    uuid: str
    title: str
    annual_gross_salary: float
    pay_day: int

    def __post_init__(self) -> None:
        _require_non_empty_text(self.uuid, "Job.uuid")
        _require_non_empty_text(self.title, "Job.title")
        _require_non_negative_float(self.annual_gross_salary, "Job.annual_gross_salary")
        _require_day_of_month(self.pay_day, "Job.pay_day")
    
    @property
    def monthly_gross_salary(self) -> float:
        return self.annual_gross_salary / 12.0
    
    @property
    def pay_schedule(self) -> CalendarMonthlySchedule:
        return CalendarMonthlySchedule(self.pay_day)
    

@dataclass(frozen=True, slots=True)
class RecurringExpense:
    uuid: str
    name: str
    amount: float
    day_of_month: int
    active: bool = True

    def __post_init__(self) -> None:
        _require_non_empty_text(self.uuid, "RecurringExpense.uuid")
        _require_non_empty_text(self.name, "RecurringExpense.name")
        _require_non_negative_float(self.amount, "RecurringExpense.amount")
        _require_day_of_month(self.day_of_month, "RecurringExpense.day_of_month")
    
    @property
    def schedule(self) -> CalendarMonthlySchedule:
        return CalendarMonthlySchedule(self.day_of_month)
    
@dataclass(frozen=True, slots=True)
class JobSearch:
    uuid: str
    started_on: datetime.date

    def __post_init__(self) -> None:
        _require_non_empty_text(self.uuid, "JobSearch.uuid")
    
@dataclass(frozen=True, slots=True)
class Employed:
    job: Job

@dataclass(frozen=True, slots=True)
class Unemployed:
    search: JobSearch | None = None

EmploymentStatus = Employed | Unemployed

@dataclass(slots=True)
class SimulationState:
    """
    Current facts only.

    State stores what is true now. It does not schedule events and does not implement domain rules.
    Processes own those rules.
    """

    cash: float
    employment: EmploymentStatus
    recurring_expenses: dict[str, RecurringExpense]
    assets: list[Any] = field(default_factory=list)
    liabilities: list[Any] = field(default_factory=list)

    monthly_job_find_probability: float = 0.3

    taxable_income_ytd: float = 0.0
    tax_paid_ytd: float = 0.0

    def __post_init__(self) -> None:
        _require_finite_float(self.cash, "SimulationState.cash")
        _require_probability(self.monthly_job_find_probability, "SimulationState.monthly_job_find_probability")
        _require_non_negative_float(self.taxable_income_ytd, "SimulationState.taxable_income_ytd")
        _require_non_negative_float(self.tax_paid_ytd, "SimulationState.tax_paid_ytd")

        if not isinstance(self.employment, (Employed, Unemployed)):
            raise TypeError("SimulationState.employment must be Employed or Unemployed.")
        
        for expense_uuid, expense in self.recurring_expenses.items():
            if expense_uuid != expense.uuid:
                raise ValueError(f"Recurring expense dictionary key must match expense.uuid. key={expense_uuid!r}, expense.uuid={expense.uuid}.")
        
    @property
    def current_job(self) -> Job | None:
        if isinstance(self.employment, Employed):
            return self.employment.job
        else:
            return None
    
    @property
    def current_job_search(self) -> JobSearch | None:
        if isinstance(self.employment, Unemployed):
            return self.employment.search
        else:
            return None
        
    @property
    def is_employed(self) -> bool:
        return isinstance(self.employment, Employed)
    
    @property
    def is_job_search_active(self) -> bool:
        return self.current_job_search is not None

### Events and event queue ###

class EventKind(enum.Enum):
    GROSS_SALARY_PAYMENT = "employment.gross_salary_payment"
    JOB_LOSS = "employment.job_loss"
    JOB_SEARCH_ATTEMPT = "employment.job_search_attempt"
    JOB_FOUND = "employment.job_found"
    SALARY_TAX_WITHHOLDING = "tax.salary_tax_withholding"
    RECURRING_EXPENSE_PAYMENT = "housing.recurring_expense_payment"

    def __str__(self) -> str:
        return self.value

class Priority(enum.IntEnum):
    """
    Lower values happen earlier on the same date.
    Same-day events scheduled by a handler must not use an earlier priority than the event currently being handled. 
    This prevents hidden time travel.
    """
    EMPLOYMENT_STATE_CHANGE = 5
    GROSS_INCOME = 10
    TAX = 20
    FIXED_EXPENSE = 30
    JOB_SEARCH_ATTEMPT = 40
    JOB_SEARCH_RESULT = 45

@dataclass(frozen=True, slots=True)
class NoPayload:
    pass

NO_PAYLOAD = NoPayload()

@dataclass(frozen=True, slots=True)
class GrossSalaryPaymentPayload:
    job_uuid: str

    def __post_init__(self) -> None:
        _require_non_empty_text(self.job_uuid, "GrossSalaryPaymentPayload.job_uuid")

@dataclass(frozen=True, slots=True)
class JobLossPayLoad:
    job_uuid: str | None = None

    def __post_init__(self) -> None:
        if self.job_uuid is not None:
            _require_non_empty_text(self.job_uuid, "JobLossPayLoad.job_uuid")


@dataclass(frozen=True, slots=True)
class JobSearchAttemptPayload:
    search_uuid: str
    
    def __post_init__(self) -> None:
        _require_non_empty_text(self.search_uuid, "JobSearchAttemptPayload.search_uuid")

@dataclass(frozen=True, slots=True)
class JobFoundPayLoad:
    search_uuid: str
    
    def __post_init__(self) -> None:
        _require_non_empty_text(self.search_uuid, "JobFoundPayLoad.search_uuid")

@dataclass(frozen=True, slots=True)
class SalaryTaxWithholdingPayload:
    gross_income: float
    source_job_uuid: str

    def __post_init__(self) -> None:
        _require_non_negative_float(self.gross_income, "SalaryTaxWithholdingPayload.gross_income")
        _require_non_empty_text(self.source_job_uuid, "SalaryTaxWithholdingPayload.source_job_uuid")



@dataclass(frozen=True, slots=True)
class RecurringExpensePaymentPayload:
    expense_uuid: str
    
    def __post_init__(self) -> None:
        _require_non_empty_text(self.expense_uuid, "RecurringExpensePaymentPayload.expense_uuid")

EventPayload = (
    NoPayload | 
    GrossSalaryPaymentPayload | JobLossPayLoad | JobSearchAttemptPayload | JobFoundPayLoad |
    SalaryTaxWithholdingPayload | RecurringExpensePaymentPayload
)

@dataclass(order=True, frozen=True, slots=True)
class Event:
    """
    Lightweight scheduled message.

    The queue orders by time, priority, sequence

    The payload is intentionally excluded from ordering.
    """
    time: datetime.date
    priority: int
    sequence: int
    kind: EventKind = field(compare=False)
    payload: EventPayload = field(default=NO_PAYLOAD, compare=False)

    def __post_init__(self) -> None:
        if not isinstance(self.time, datetime.date):
            raise TypeError(f"Event.time must be a datetime.date. Got {type(self.time)}.")
        if not isinstance(self.kind, EventKind):
            raise TypeError(f"Event.kind must be an EventKind. Got {type(self.kind)}.")
        _require_positive_int(self.sequence, "Event.sequence")

PayloadT = TypeVar("PayloadT")

def expect_payload(event: Event, payload_type: type[PayloadT]) -> PayloadT:
    """
    Runtime payload assertion with a clear error message.

    This keeps events lightweight while avoiding unstructured dictionary payload inside handlers.
    """
    if not isinstance(event.payload, payload_type):
        raise TypeError(f"Event {event.kind.value!r} expected payload {payload_type.__name__}, got {type(event.payload).__name__}.")
    return event.payload

class EventQueue:
    """
    Priority Queue wrapper.

    This class prevents the simulation loop from accidentally popping and discarding future events when running to an intermediate date.
    """

    def __init__(self) -> None:
        self._heap: list[Event] = []
    
    def push(self, event: Event) -> None:
        heapq.heappush(self._heap, event)
    
    def pop_next(self) -> Event:
        if not self._heap:
            raise IndexError("Cannot pop from an empty EventQueue.")
        return heapq.heappop(self._heap)
    
    def peek(self) -> Event | None:
        if not self._heap:
            return None
        return self._heap[0]

    def snapshot(self) -> tuple[Event, ...]:
        return tuple(sorted(self._heap))
    
    def __len__(self) -> int:
        return len(self._heap)
    
    def __bool__(self) -> bool:
        return bool(self._heap)

### Simulation Engine ###

class LogLevel(enum.Enum):
    LEDGER = "ledger"
    INFO = "info"
    DEBUG = "debug"
    WARNING = "warning"

@dataclass(frozen=True, slots=True)
class LogEntry:
    time: datetime.date
    level: LogLevel
    message: str
    cash: float
    cash_delta: float | None = None
    event_kind: EventKind | None = None

Handler = Callable[[Event, "Simulation"], None]


class Process(Protocol):
    """
    A process owns a familiy of event types.
    Each process has two responsibilities:
    - initialize(sim): schedule first relevant events
    - handlers(): declare the event kind it handles
    """

    def initialize(self, sim: "Simulation") -> None: ...    
    def handlers(self) -> Mapping[EventKind, Handler]: ...


class Simulation:
    """
    Discrete-event simulation engine.

    Owns:
    - state
    - event queue
    - processes
    - dispatch table
    - random number generator
    - log
    """

    def __init__(self, state: SimulationState, start_date: datetime.date, seed: int = 0):
        if not isinstance(start_date, datetime.date):
            raise TypeError(f"start_date must be a datetime.date. Got {type(start_date)}.")
        
        self.state = state
        self.start_date = start_date
        self.now = start_date
        self.rng = random.Random(seed)

        self._queue = EventQueue()
        self._sequence = 0
        
        self._processes: list[Process] = []
        self._handlers: dict[EventKind, Handler] = {}

        self._initialized = False
        self._initializing = False
        self._current_event: Event | None = None

        self.log: list[LogEntry] = []
    
    @property
    def pending_events(self) -> tuple[Event, ...]:
        return self._queue.snapshot()
    
    def register_process(self, process: Process) -> None:
        if self._initialized or self._initializing:
            raise RuntimeError("Cannot register processes after initialization.")
        
        handlers = dict(process.handlers())

        for event_kind, handler in handlers.items():
            if not isinstance(event_kind, EventKind):
                raise TypeError(f"Process handler keys must be EventKind values. Got {type(event_kind)}.")
            if event_kind in self._handlers:
                raise ValueError(f"Handler already registered for event kind {event_kind.value!r}.")
            
            self._handlers[event_kind] = handler
        
        self._processes.append(process)

    def initialize(self) -> None:
        """
        Schedule initial events once.

        Calling initialize repeatdly is safe: subsequent calls do nothing.
        """
        if self._initialized:
            return
        if self._initializing:
            raise RuntimeError("Simulation initialization is already in progress.")
        self._initializing = True
        
        try:
            for process in self._processes:
                process.initialize(self)
        finally:
            self._initializing = False
        
        self._initialized = True
    
    def schedule(self, time: datetime.date, kind: EventKind, priority: Priority, payload: EventPayload = NO_PAYLOAD):
        """
        Add a new event to the queue.
        """

        if not isinstance(time, datetime.date):
            raise TypeError(f"Scheduled event time must be a datetime.date. Got {type(time)}.")
        if not isinstance(kind, EventKind):
            raise TypeError(f"Scheduled event kind must be a EventKind. Got {type(time)}.")
        if kind not in self._handlers:
            raise KeyError(f"Cannot schedule {kind.value!r}: no process registered a handler.")
        if time < self.now:
            raise ValueError(f"Cannot schedule {kind.value!r} for {time}, as simulation time is already {self.now}.")
        priority_value = int(priority)

        if self._current_event is not None:
            if time == self.now and priority_value < self._current_event.priority:
                msg = "Cannot schedule a same-day event with an earlier priority than the event currently being handled. "
                msg += "Current event={self._current_event.kind.value!r} priority={self._current_event.kind.priority}, "
                msg += "new event={kind.value!r} priority={priority_value}."
                raise ValueError(msg)

        self._sequence += 1
        self._queue.push(Event(time=time, priority=priority_value, sequence=self._sequence, kind=kind, payload=payload))

    def record(self, message: str, *, level: LogLevel = LogLevel.INFO, cash_delta: float | None = None) -> None:
        self.log.append(
            LogEntry(
                time=self.now, level=level, message=message, cash=self.state.cash, cash_delta=cash_delta,
                event_kind=(self._current_event.kind if self._current_event else None)
            )
        )
    
    def adjust_cash(self, delta: float, message: str, *, level: LogLevel = LogLevel.LEDGER) -> None:
        _require_finite_float(delta, "cash delta")
        self.state.cash += float(delta)
        self.record(message, level=level, cash_delta=float(delta))
        
    def run_until(self, end_date: datetime.date) -> None:
        """
        Run all events with event.time <= end_date.

        Future events remain in the queue, so this method can be called repeatedly with increasing dates.
        """
        if not isinstance(end_date, datetime.date):
            raise TypeError(f"end_date must be a datetime.date. Got {type(end_date)}.")
        if end_date < self.now:
            raise ValueError(f"Cannot run backwards from {self.now} to {end_date}.")
        
        self.initialize()

        while True:
            next_event = self._queue.peek()
            if next_event is None or next_event.time > end_date:
                break

            event = self._queue.pop_next()
            self.now = event.time
            handler = self._handlers.get(event.kind)
            if handler is None:
                raise KeyError(f"No handler registered for event kind {event.kind.value!r}.")
            previous_event = self._current_event
            self._current_event = event
            try:
                handler(event, self)
            finally:
                self._current_event = previous_event
        
        self.now = end_date

@dataclass(frozen=True, slots=True)
class ScheduledJobLoss:
    time: datetime.date
    job_id: str | None = None

    def __post_init__(self) -> None:
        if not isinstance(self.time, datetime.date):
            raise TypeError("ScheduledJobLoss.time must be a datetime.date")
        if self.job_id is not None:
            _require_non_empty_text(self.job_id, "ScheduledJobLoss.job_id")
    
@dataclass(frozen=True, slots=True)
class ReplacementJobPolicy:
    title: str
    annual_gross_salary: float
    pay_day: int
    id_prefix: str = "replacement-job"

    def __post_init__(self) -> None:
        _require_non_empty_text(self.title, "ReplacementJobPolicy.title")
        _require_non_negative_float(self.annual_gross_salary, "ReplacementJobPolicy.annual_gross_salary")
        _require_day_of_month(self.pay_day, "ReplacementJobPolicy.pay_day")
        _require_non_empty_text(self.id_prefix, "ReplacementJobPolicy.id_prefix")
    
    def create_job(self, sequence: int) -> Job:
        _require_positive_int(sequence, "replacement job sequence")
        return Job(uuid=f"{self.id_prefix}-{sequence}", title=self.title, annual_gross_salary=self.annual_gross_salary, pay_day=self.pay_day)
    

class EmploymentProcess:
    """
    Owns:
        - salary payments
        - job loss
        - job-search attempts
        - job-found transitions
    It does not own tax calculation.
    It schedules a tax event after gross salary is paid, and TaxProcess handles that event.    
    """

    def __init__(
        self, 
        *, 
        scheduled_job_losses: Iterable[ScheduledJobLoss], 
        replacement_job_policy: ReplacementJobPolicy, 
        search_interval: FixedDayIntervalSchedule = FixedDayIntervalSchedule(days=30),
    ) -> None:
        
        self.scheduled_job_losses = tuple(scheduled_job_losses)
        self.replacement_job_policy = replacement_job_policy
        self.search_interval = search_interval
        
        self._job_search_sequence = 0
        self._replacement_job_sequence = 0

    def initialize(self, sim: Simulation) -> None:
        current_job = sim.state.current_job

        if current_job is not None:
            first_pay_date = current_job.pay_schedule.first_on_or_after(sim.start_date)
            sim.schedule(first_pay_date, EventKind.GROSS_SALARY_PAYMENT, Priority.GROSS_INCOME, GrossSalaryPaymentPayload(job_uuid=current_job.uuid))
        
        for job_loss in self.scheduled_job_losses:
            if job_loss.time < sim.start_date:
                continue
            
            sim.schedule(job_loss.time, EventKind.JOB_LOSS, Priority.EMPLOYMENT_STATE_CHANGE, JobLossPayLoad(job_uuid=job_loss.job_id))
        
    def handlers(self) -> Mapping[EventKind, Handler]:
        return {
            EventKind.GROSS_SALARY_PAYMENT: self.handle_gross_salary_payment, 
            EventKind.JOB_LOSS: self.handle_job_loss, 
            EventKind.JOB_SEARCH_ATTEMPT: self.handle_job_search_attempt,
            EventKind.JOB_FOUND: self.handle_job_found
        }
        
    def handle_gross_salary_payment(self, event: Event, sim: Simulation) -> None:
        payload = expect_payload(event, GrossSalaryPaymentPayload)
        current_job = sim.state.current_job

        if current_job is None or current_job.uuid != payload.job_uuid:
            sim.record(f"Gross salary ignored for stale job_uuid={payload.job_uuid!r}", level=LogLevel.DEBUG)
            return

        gross = current_job.monthly_gross_salary
        sim.state.taxable_income_ytd += gross
        sim.adjust_cash(gross, f"Gross salary paid from {current_job.title}: +GBP {gross:,.0f}")
        sim.schedule(
            event.time, EventKind.SALARY_TAX_WITHHOLDING, Priority.TAX, SalaryTaxWithholdingPayload(gross_income=gross, source_job_uuid=current_job.uuid)
        )
        sim.schedule(
            current_job.pay_schedule.next_after(event.time), EventKind.GROSS_SALARY_PAYMENT, Priority.GROSS_INCOME, GrossSalaryPaymentPayload(job_uuid=current_job.uuid)
        )
    
    def handle_job_loss(self, event: Event, sim: Simulation) -> None:
        payload = expect_payload(event, JobLossPayLoad)

        if not isinstance(sim.state.employment, Employed):
            sim.record("Job loss ignored because there is no current job", level=LogLevel.DEBUG)
            return
        
        old_job = sim.state.employment.job
        
        if payload.job_uuid is not None and old_job.uuid != payload.job_uuid:
            sim.record(f"Job loss ignored because current job is not {payload.job_uuid!r}", level=LogLevel.DEBUG)
            return
        
        self._job_search_sequence += 1
        search = JobSearch(uuid=str(self._job_search_sequence), started_on=event.time)
        sim.state.employment = Unemployed(search=search)

        sim.record(f"Lost job: {old_job.title}. Started job search")
        sim.schedule(self.search_interval.next_after(event.time), EventKind.JOB_SEARCH_ATTEMPT, Priority.JOB_SEARCH_ATTEMPT, JobSearchAttemptPayload(search_uuid=search.uuid))

    def handle_job_search_attempt(self, event: Event, sim: Simulation) -> None:
        payload = expect_payload(event, JobSearchAttemptPayload)
        current_search = sim.state.current_job_search
        
        if current_search is None or current_search.uuid != payload.search_uuid:
            sim.record(f"Job search ignored for stale search_id={payload.search_uuid}", level=LogLevel.DEBUG)
            return
        
        probability = sim.state.monthly_job_find_probability
        draw = sim.rng.random()
        if draw <= probability:
            sim.record(f"Job search succeeded: random draw {draw:.2f} <= {probability:.2f}")
            sim.schedule(event.time, EventKind.JOB_FOUND, Priority.JOB_SEARCH_RESULT, JobFoundPayLoad(search_uuid=current_search.uuid))
            return

        sim.record(f"Job search failed: random draw {draw:.2f} > {probability:.2f}")
        sim.schedule(self.search_interval.next_after(event.time), EventKind.JOB_SEARCH_ATTEMPT, Priority.JOB_SEARCH_ATTEMPT, JobSearchAttemptPayload(search_uuid=current_search.uuid))

    def handle_job_found(self, event: Event, sim: Simulation) -> None:
        payload = expect_payload(event, JobFoundPayLoad)
        current_search = sim.state.current_job_search

        if current_search is None or current_search.uuid != payload.search_uuid:
            sim.record(f"Job found ignored for stale search_uuid={payload.search_uuid}", level=LogLevel.DEBUG)
            return
        
        self._replacement_job_sequence += 1
        new_job = self.replacement_job_policy.create_job(self._replacement_job_sequence)
        sim.state.employment = Employed(job=new_job)
        sim.record(f"Found job: {new_job.title}, gross salary GBP {new_job.annual_gross_salary:,.0f}")

        first_pay_date = new_job.pay_schedule.first_after(event.time)
        sim.schedule(first_pay_date, EventKind.GROSS_SALARY_PAYMENT, Priority.GROSS_INCOME, GrossSalaryPaymentPayload(job_uuid=new_job.uuid))


class TaxProcess:
    """
    Owns tax events.

    This implementation uses a flat salary tax rate. It is deliberately isolated so it can later be replaced with progressive brackets, 
    allowances, national insurance, pension deductions, annual settlement, or refunds without changing the employement process.
    """

    def __init__(self, *, flat_salary_tax_rate: float):
        _require_probability(flat_salary_tax_rate, "flat_salary_tax_rate")
        self.flat_salary_tax_rate = flat_salary_tax_rate
    
    def initialize(self, sim: Simulation) -> None:
        return None
    
    def handlers(self) -> Mapping[EventKind, Handler]:
        return {EventKind.SALARY_TAX_WITHHOLDING: self.handle_salary_tax_withholding}
    
    def handle_salary_tax_withholding(self, event: Event, sim: Simulation) -> None:
        payload = expect_payload(event, SalaryTaxWithholdingPayload)
        tax = payload.gross_income * self.flat_salary_tax_rate
        sim.state.tax_paid_ytd += tax
        sim.adjust_cash(-tax, f"Salary tax withheld: - GBP {tax:,.0f}")
    

class HousingProcess:
    """
    Owns recurring housing and fixed-expense payments.

    This can handle rent today and later be extended to mortgage payments, service charges, council tax, 
    utilities, subscriptions, or other fixed recurring obligations.
    """

    def initialize(self, sim: Simulation) -> None:
        for expense in sim.state.recurring_expenses.values():
            if not expense.active:
                continue

            sim.schedule(
                expense.schedule.first_on_or_after(sim.start_date), EventKind.RECURRING_EXPENSE_PAYMENT, 
                Priority.FIXED_EXPENSE, RecurringExpensePaymentPayload(expense_uuid=expense.uuid)
            )
    
    def handlers(self) -> Mapping[EventKind, Handler]:
        return {EventKind.RECURRING_EXPENSE_PAYMENT: self.handle_recurring_expense_payment}
    
    def handle_recurring_expense_payment(self, event: Event, sim: Simulation) -> None:
        payload = expect_payload(event, RecurringExpensePaymentPayload)
        expense = sim.state.recurring_expenses.get(payload.expense_uuid)

        if expense is None:
            sim.record(f"Recurring expense ignored because {payload.expense_uuid!r} does not exist", level=LogLevel.WARNING)
            return
        
        if not expense.active:
            sim.record(f"Recurring expense ignored because {expense.name!r} is inactive", level=LogLevel.DEBUG)
            return
        
        sim.adjust_cash(-expense.amount, f"{expense.name} paid: - GBP {expense.amount:,.0f}")
        sim.schedule(expense.schedule.next_after(event.time), EventKind.RECURRING_EXPENSE_PAYMENT, Priority.FIXED_EXPENSE, RecurringExpensePaymentPayload(expense_uuid=expense.uuid))


def format_money(amount: float) -> str:
    return f"£ {amount:,.0f}"

def print_log(sim: Simulation, *, include_debug: bool = False) -> None:
    for entry in sim.log:
        if entry.level is LogLevel.DEBUG and not include_debug:
            continue

        print(f"{entry.time} | {entry.level.value:<7} | {entry.message:<78} | cash = {format_money(entry.cash)}")
    
    print()
    print(f"Final cash: {format_money(sim.state.cash)}")
    print(f"Taxable income YTD: {format_money(sim.state.taxable_income_ytd)}")
    print(f"Tax paid YTD: {format_money(sim.state.tax_paid_ytd)}")
    print(f"Pending future events: {len(sim.pending_events)}")


state = SimulationState(
    cash=40_000.0,
    employment=Employed(job=Job(uuid="initial-job", title="Quant researcher", annual_gross_salary=55_000.0, pay_day=24)),
    recurring_expenses={
        "rent": RecurringExpense(uuid="rent", name="Rent", amount=600.0, day_of_month=24),
        # "bills": RecurringExpense(uuid="bills", name="Bills", amount=142.86, day_of_month=15),
        "lifestyle": RecurringExpense(uuid="lifestyle", name="Lifestyle", amount=1_250.0, day_of_month=30)
    }
)

sim = Simulation(
    state=state,
    start_date=datetime.date(2026, 6, 21),
    seed = 69
)
sim.register_process(EmploymentProcess(
    scheduled_job_losses=[ScheduledJobLoss(time=datetime.date(2027, 3, 1))], 
    # scheduled_job_losses = [],
    replacement_job_policy=ReplacementJobPolicy(title="Replacement full-time job", annual_gross_salary=70_000.0, pay_day=24)
))
sim.register_process(TaxProcess(flat_salary_tax_rate=0.41))
sim.register_process(HousingProcess())

sim.run_until(datetime.date(2030, 1, 1))
print_log(sim, include_debug=True)

2026-06-24 | ledger  | Gross salary paid from Quant researcher: +GBP 4,583                            | cash = £ 44,583
2026-06-24 | ledger  | Salary tax withheld: - GBP 1,879                                               | cash = £ 42,704
2026-06-24 | ledger  | Rent paid: - GBP 600                                                           | cash = £ 42,104
2026-06-30 | ledger  | Lifestyle paid: - GBP 1,250                                                    | cash = £ 40,854
2026-07-24 | ledger  | Gross salary paid from Quant researcher: +GBP 4,583                            | cash = £ 45,438
2026-07-24 | ledger  | Salary tax withheld: - GBP 1,879                                               | cash = £ 43,558
2026-07-24 | ledger  | Rent paid: - GBP 600                                                           | cash = £ 42,958
2026-07-30 | ledger  | Lifestyle paid: - GBP 1,250                                                    | cash = £ 41,708
2026-08-24 | ledger  | Gross salary paid

2026-06-24 | ledger  | Gross salary paid from Quant researcher: +GBP 4,583                            | cash = £ 44,583
2026-06-24 | ledger  | Salary tax withheld: - GBP 1,879                                               | cash = £ 42,704
2026-06-24 | ledger  | Rent paid: - GBP 600                                                           | cash = £ 42,104
2026-06-30 | ledger  | Lifestyle paid: - GBP 1,250                                                    | cash = £ 40,854
2026-07-24 | ledger  | Gross salary paid from Quant researcher: +GBP 4,583                            | cash = £ 45,438
2026-07-24 | ledger  | Salary tax withheld: - GBP 1,879                                               | cash = £ 43,558
2026-07-24 | ledger  | Rent paid: - GBP 600                                                           | cash = £ 42,958
2026-07-30 | ledger  | Lifestyle paid: - GBP 1,250                                                    | cash = £ 41,708
2026-08-24 | ledger  | Gross salary paid

In [12]:
# state = SimulationState(
#     cash=55_000.0,
#     employment=Employed(job=Job(uuid="initial-job", title="Quant researcher at QRT", annual_gross_salary=107_000.0, pay_day=25)),
#     recurring_expenses={
#         "rent": RecurringExpense(uuid="rent", name="Rent", amount=1_100.0, day_of_month=1),
#         "bills": RecurringExpense(uuid="bills", name="Bills", amount=142.86, day_of_month=15),
#         # "lifestyle": RecurringExpense(uuid="lifestyle", name="Lifestyle", amount=1_500.0, day_of_month=30)
#     }
# )

# sim = Simulation(
#     state=state,
#     start_date=datetime.date(2026, 6, 20),
#     seed = 69
# )
# sim.register_process(EmploymentProcess(scheduled_job_losses=[], replacement_job_policy=ReplacementJobPolicy(title="Replacement full-time job", annual_gross_salary=1_000_000.0, pay_day=21)))
# sim.register_process(TaxProcess(flat_salary_tax_rate=0.4))
# sim.register_process(HousingProcess())

# sim.run_until(datetime.date(2030, 1, 1))
# print_log(sim, include_debug=True)

In [10]:
sim.pending_events

(Event(time=datetime.date(2030, 1, 15), priority=30, sequence=171, kind=<EventKind.RECURRING_EXPENSE_PAYMENT: 'housing.recurring_expense_payment'>, payload=RecurringExpensePaymentPayload(expense_uuid='bills')),
 Event(time=datetime.date(2030, 1, 25), priority=10, sequence=173, kind=<EventKind.GROSS_SALARY_PAYMENT: 'employment.gross_salary_payment'>, payload=GrossSalaryPaymentPayload(job_uuid='initial-job')),
 Event(time=datetime.date(2030, 2, 1), priority=30, sequence=174, kind=<EventKind.RECURRING_EXPENSE_PAYMENT: 'housing.recurring_expense_payment'>, payload=RecurringExpensePaymentPayload(expense_uuid='rent')))

In [11]:
Module Account
Module Company


SyntaxError: invalid syntax (3081735691.py, line 1)